## InsureLLM expert Q&A (RAG chat)

### Retrieval + LLM + Gradio

This notebook is **Part 2** of the InsureLLM RAG lab in `13_LANGCHAIN`:

1. **First** run [`rag_langchain_chunks_vector_db_visualization.ipynb`](./rag_langchain_chunks_vector_db_visualization.ipynb) — load `knowledge-base/`, chunk, embed with OpenAI `text-embedding-3-small`, persist **`vector_db/`**.
2. **Then** run this notebook — open that Chroma DB, retrieve relevant chunks for a question, pass them to the LLM, optionally chat in Gradio.

No local **sentence-transformers** / Hugging Face embeddings — same API-only setup as the chunks notebook.

### How to read

Each code cell has a markdown cell above it: fenced code block + **line-by-line** table (same style as the chunks notebook).

### Pipeline in this notebook

```text
User question → embed query (OpenAI) → Chroma similarity search → top-k chunks
       → system prompt + context → ChatOpenAI → answer
       → (optional) Gradio ChatInterface
```

Run with working directory **`13_LANGCHAIN`** so `vector_db/` resolves correctly.

### Code cell — imports

```python
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr
```

| Line / piece | Meaning |
|--------------|---------|
| `import os` | Standard library — read `OPENAI_API_KEY` from the environment after `load_dotenv`. |
| `from dotenv import load_dotenv` | Package **python-dotenv** — load secrets from a `.env` file in this folder. |
| `from langchain_openai import ChatOpenAI, OpenAIEmbeddings` | Package **langchain-openai** — chat model + embedding API wrappers. |
| `from langchain_chroma import Chroma` | Package **langchain-chroma** — connect to the on-disk Chroma DB created in the chunks notebook. |
| `from langchain_core.messages import SystemMessage, HumanMessage` | Package **langchain-core** — message types for multi-turn / system+user prompts to the chat model. |
| `import gradio as gr` | Package **gradio** — quick web UI (`ChatInterface`) for testing the bot. |

**Install (once per env):** `pip install langchain-openai langchain-chroma langchain-core python-dotenv gradio chromadb`

**Prerequisite:** `vector_db/` must already exist from the chunks notebook (same embedding model).

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

/home/soudas/miniforge3/envs/my-proj/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Code cell — configuration and API key

```python
MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
```

#### Purpose

Set the **chat** model name and the Chroma folder **before** opening the vector store. The **embedding** model is configured in the next cell and must match what you used when building `vector_db/`.

| Line | Meaning |
|------|---------|
| `MODEL = "gpt-4.1-nano"` | Low-cost OpenAI chat model for answers (separate from `text-embedding-3-small`). |
| `db_name = "vector_db"` | Same folder name as in the chunks notebook — persisted Chroma data under `13_LANGCHAIN/vector_db/`. |
| `load_dotenv(override=True)` | Load `.env`; `override=True` replaces existing env vars so the file wins. |
| `os.getenv("OPENAI_API_KEY")` | Key used by **both** `OpenAIEmbeddings` (query vectors) and `ChatOpenAI` (generation). |
| `print` first 8 chars | Sanity check only — do not print full keys in shared notebooks. |

Put `.env` next to this notebook: `OPENAI_API_KEY=sk-...`

In [4]:
MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


### Code cell — connect to Chroma (load existing vectors)

```python
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)
```

#### Purpose

Open the **existing** index on disk. This cell does **not** re-chunk or re-embed all documents (that was Part B in the chunks notebook). It only needs the **same** embedding client so **new questions** are embedded in the same 1536-dimensional space as the stored chunk vectors.

#### Line-by-line

| Line | Meaning |
|------|---------|
| `embeddings = OpenAIEmbeddings(model="text-embedding-3-small")` | Embedding client — must match the model used in `Chroma.from_documents(...)` when you built `vector_db/`. Mismatch → retrieval quality collapses. |
| `vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)` | Attach LangChain to the Chroma files in `vector_db/`. `embedding_function` is used when you search (embed the query) or add new docs later. |

#### What is in `vector_db/`?

For each chunk: **text**, **metadata** (e.g. source file), and a **vector**. Chroma runs similarity search between the question vector and chunk vectors.

If you get an empty store or errors, re-run the **embed + Chroma** cells in the chunks notebook first.

In [5]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)

### Code cell — retriever and chat LLM

```python
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)
```

#### Purpose

Two objects drive RAG here:

- **Retriever** — turns a question string into the top-k similar `Document` chunks from Chroma.
- **LLM** — reads your system prompt (with retrieved context) and writes the final answer.

#### Line-by-line

| Line | Meaning |
|------|---------|
| `retriever = vectorstore.as_retriever()` | Wraps the vector store as a LangChain **Retriever** (default `search_kwargs` often `k=4` — four chunks per question unless you change it). |
| `llm = ChatOpenAI(temperature=0, model_name=MODEL)` | Chat client using `MODEL`; `temperature=0` favors the highest-probability tokens (more deterministic answers). |

---

#### What is `as_retriever()`?

`as_retriever()` turns a **vector store** into a **Retriever** — an object whose job is: **given a question string, return relevant `Document` chunks** (not an LLM answer).

**What you have before**

```python
vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)
```

`vectorstore` knows how to:

- store vectors + text in Chroma
- run **similarity search** (embed the query, find nearest chunk vectors)

That API is “database-ish”: you often call things like `similarity_search("Who is Avery?", k=4)`.

**What `as_retriever()` does**

```python
retriever = vectorstore.as_retriever()
docs = retriever.invoke(question)
```

It wraps the same store in the **Retriever** interface LangChain uses everywhere in RAG:

- one main method: **`retriever.invoke(question)`** (or `.get_relevant_documents(question)` in older code)
- returns a **list of `Document`** objects (`page_content` + `metadata`)

Means: embed `question` → Chroma top‑k similar chunks → those chunks as `Document`s.

**Why not call Chroma directly?**

1. **Same shape as other retrievers** — keyword, hybrid, multi-query, etc. all implement “question → docs”. Chains and tools expect that.
2. **Composable** — swap `vectorstore.as_retriever()` for `vectorstore.as_retriever(search_kwargs={"k": 8})` or a different retriever without rewriting most of your Q&A logic.
3. **Fits LangChain RAG patterns** — retriever → stuff context into prompt → LLM.

**Default behavior**

`as_retriever()` with no arguments typically uses something like **`k=4`**: four chunks per question. Change it with:

```python
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})
```

Other `search_kwargs` depend on the vector store (e.g. score threshold, metadata filters).

**Mental model**

| Object | Role |
|--------|------|
| `OpenAIEmbeddings` | text → vector (for queries and, at index time, for chunks) |
| `Chroma` / `vectorstore` | **store** vectors and run similarity search |
| `retriever` | **thin adapter**: “user question” → “best matching documents” |
| `ChatOpenAI` | reads those docs in the prompt and **writes** the answer |

```text
question
   → retriever.invoke(question)     # retrieval only
   → [Document, Document, ...]
   → join page_content → context
   → LLM with system + human messages
   → answer string
```

`retriever.invoke(question)` is the **R** in RAG. Without it you’d only have `llm.invoke(question)` — no guaranteed grounding in `knowledge-base/`.

Also in notes: [`6_indexes_loaders_retrievers_vector_stores.md`](./6_indexes_loaders_retrievers_vector_stores.md).

---

#### Sidebar on `temperature`

- Controls how random token sampling is during generation.
- `temperature=0` → usually pick the single most likely next token each step (still not always perfectly reproducible without a fixed seed).
- Higher temperature → more varied wording.
- For **tone or creativity**, prefer changing the **system prompt**, not only temperature.

In [9]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

### Code cell — try the retriever alone (`invoke`)

```python
retriever.invoke("Who is Avery?")
```

#### Purpose

Inspect **retrieval only** — no LLM yet. You should get a **list** of LangChain `Document` objects whose `page_content` mentions Avery (or related InsureLLM employees), depending on chunking and similarity.

#### Line-by-line

| Line | Meaning |
|------|---------|
| `retriever.invoke("Who is Avery?")` | Embed the question with `text-embedding-3-small`, query Chroma, return top-k matching chunks. Same entry point Gradio will use indirectly via `answer_question`. |

If results look wrong, check: (1) `vector_db/` built from `knowledge-base/`, (2) embedding model matches, (3) spelling in the question (try full name from an employee file).

---

#### `retriever.invoke` vs `llm.invoke` (same method name, different job)

| | `retriever.invoke("Who is Avery?")` | `llm.invoke("Who is Avery?")` |
|--|------------------------------------|-------------------------------|
| **What it does** | **Search** your vector store | **Generate** text with a chat model |
| **Talks to** | Chroma (+ embedding API for the query) | OpenAI chat API (`ChatOpenAI`) |
| **Uses your knowledge base?** | **Yes** — finds chunks in `vector_db/` | **No** — only the model’s training knowledge |
| **Returns** | `list[Document]` (chunk text + metadata) | `AIMessage` (natural-language answer) |
| **Writes a final answer?** | No — only returns source text | Yes — but may be wrong / generic for InsureLLM |
| **RAG role** | **R**etrieval | **G**eneration (when given context) |

```python
# Retrieval only — no prose answer
docs = retriever.invoke("Who is Avery?")
# → [Document(... Avery Lancaster ...), ...]

# Generation only — no company docs
msg = llm.invoke("Who is Avery?")
# → AIMessage(content="Avery is a given name...")  # generic
```

Full RAG uses **both**: retrieve docs → put them in `{context}` → `llm.invoke([SystemMessage(...), HumanMessage(...)])`.

- **`retriever.invoke`** = “find me relevant passages”
- **`llm.invoke`** = “write an answer from what I give you (or from what you already know)”

Also in notes: [`6_indexes_loaders_retrievers_vector_stores.md`](./6_indexes_loaders_retrievers_vector_stores.md).

In [10]:
retriever.invoke("Who is Avery?")

[Document(id='84265495-6c9d-420e-9c3a-1a6b6e43a8c0', metadata={'source': 'knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content='# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000  \n\n## Insurellm Career Progression\n- **2015 - Present**: Co-Founder & CEO  \n  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  \n\n- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  \n  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the t

### Code cell — try the LLM alone (`invoke`)

```python
llm.invoke("Who is Avery?")
```

#### Purpose

Shows what the model does **without RAG** — it only knows training data, not your `knowledge-base/`. Compare this answer to the RAG pipeline later: RAG should ground answers in retrieved chunks.

#### Line-by-line

| Line | Meaning |
|------|---------|
| `llm.invoke("Who is Avery?")` | Sends a plain string; LangChain treats it as a user message. Returns an `AIMessage` (InsureLLM-specific facts may be missing or hallucinated). |

Contrast with the previous cell: **retriever** returned Avery Lancaster **Documents** from disk; **llm** alone often answers that Avery is “a given name…” — no InsureLLM CEO context.

In the full app we pass `[SystemMessage(...), HumanMessage(...)]` instead of a bare string.

In [11]:
llm.invoke("Who is Avery?")

AIMessage(content='Could you please provide more context or specify which Avery you are referring to? There are many individuals and characters named Avery.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 11, 'total_tokens': 35, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_109d5ede69', 'id': 'chatcmpl-ESgDQfJhqwVwteE3UW0OvKOWnRRdi', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0e25d-6a58-7120-b2c7-b7a05b1d5064-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 24, 'total_tokens': 35, 'input_tok

## Put retrieval and generation together

### Code cell — system prompt template

```python
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""
```

#### Line-by-line

| Line | Meaning |
|------|---------|
| Triple-quoted string | Multi-line template stored in Python; not sent to the API until formatted. |
| Role instructions | Tell the model it represents InsureLLM and should stay honest if context is insufficient. |
| `{context}` placeholder | Filled per question with text from retrieved chunks (`str.format(context=...)`). |

**RAG pattern:** retrieved text lives in the **system** message (or a dedicated user block); the user message stays the actual question.

In [12]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

### Code cell — `answer_question` (full RAG step)

```python
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content
```

#### Line-by-line

| Line | Meaning |
|------|---------|
| `def answer_question(question: str, history):` | Function Gradio `ChatInterface` calls: `question` is the latest user text; `history` is prior turns (unused here but required by the Gradio signature). |
| `docs = retriever.invoke(question)` | **Retrieve** — list of `Document` chunks from Chroma for this question. |
| `context = "\n\n".join(...)` | Concatenate chunk bodies with blank lines so the model can tell chunk boundaries. |
| `system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)` | Inject retrieved text into the system template. |
| `llm.invoke([SystemMessage(...), HumanMessage(...)])` | **Generate** — system holds context + rules; human holds only the user question. |
| `return response.content` | Plain string answer for Gradio / `print` (strip of `AIMessage` wrapper). |

```text
question → retriever → chunks → context string → system prompt → LLM → answer
```

In [13]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

### Code cell — smoke test (one question)

```python
answer_question("Who is Avery Lancaster?", [])
```

#### Line-by-line

| Line | Meaning |
|------|---------|
| `answer_question("Who is Avery Lancaster?", [])` | Run the full RAG path once; `[]` is empty chat history. Expect an answer grounded in employee markdown from `knowledge-base/employees/`. |

Typo test: course example used "Averi" — correct spelling **Avery** matches the knowledge base filename.

In [14]:
answer_question("Who is Avery Lancaster?", [])

"Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm, an insurance technology company she co-founded in 2015. She is based in San Francisco, California, and has played a key role in guiding the company's growth and innovation in the insurance industry. Avery has a background in product management and has been recognized for her leadership, risk management expertise, and commitment to diversity and community engagement."

### Code cell — Gradio chat UI

```python
gr.ChatInterface(answer_question).launch()
```

#### Line-by-line

| Line | Meaning |
|------|---------|
| `gr.ChatInterface(answer_question)` | Wraps your function in a chat UI; Gradio passes `(message, history)` on each send. |
| `.launch()` | Starts a local web server (URL printed in the notebook output). Each message runs retrieve → LLM again. |

Stop the server with the notebook interrupt control when finished.

---

That is the core RAG loop: **embeddings already in Chroma** → **retrieve** → **augment prompt** → **generate**. See also [`10_embeddings_and_vector_store.md`](./10_embeddings_and_vector_store.md) and [`6_indexes_loaders_retrievers_vector_stores.md`](./6_indexes_loaders_retrievers_vector_stores.md).

In [16]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


/home/soudas/miniforge3/envs/my-proj/lib/python3.14/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/home/soudas/miniforge3/envs/my-proj/lib/python3.14/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
